# SHROOM OOTB Baseline Runner — Colab VS Code Extension Workflow

Recommended use with the **Google Colab VS Code extension**:
1. Start/connect to a Colab GPU runtime.
2. Run the cells top-to-bottom.
3. Begin with `DRY_RUN = True` or `ONLY_CONTAINS = "flan-t5-small"`.
4. Then run by family or full batch.

This cleaned version assumes your project files already handle device selection internally, e.g. your model classes use CUDA when available. The notebook does **not** patch or rewrite `src/` model files. It only copies the Drive project to `/content`, runs your existing `run_experiment.py` for each model, and backs up outputs/logs to Google Drive after each model.


In [1]:
# 1) Configuration
from pathlib import Path
from datetime import datetime

# Google Drive project folder containing run_experiment.py, src/, data/, participant_kit/.
DRIVE_PROJECT = Path("/content/drive/MyDrive/thesis_colab/model_experiments_colab")

# Temporary Colab workspace used for actual execution.
LOCAL_PROJECT = Path("/content/model_experiments_colab")

# Where this runner backs up outputs and logs in Google Drive.
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/thesis_colab/outputs_all_baselines_colab_vscode")

# One timestamped backup folder for this session.
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_RUN_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / RUN_TAG

# Dataset/scoring paths relative to LOCAL_PROJECT.
INPUT_PATH = "data/SHROOM_dev-v2/val.model-agnostic.json"
WARMUP_PATH = "data/SHROOM_trial-v1.1/trial-v1.json"
WARMUP_N = 10

SEED = 42
PREVIEW_N = 2
PROMPT_VERSION = "support_prompt_v1"

# Execution controls.
CONTINUE_ON_ERROR = True # Use this to decide whether one failed model stops the whole batch.
SKIP_EXISTING = True # Use this to avoid rerunning models that already have a metadata file.
DRY_RUN = False # Use this to check what would run without actually running models (if set to True); prints the selected models/commands but does not execute run_experiment.py.

# Optional filters. Examples:
# ONLY_TYPES = ["flan"]
# ONLY_CONTAINS = "Qwen2.5-0.5B"
ONLY_TYPES = ["qwen"] # Use this to run only one or more model families.
ONLY_CONTAINS = "7B" # Use this to run only models whose Hugging Face name contains a specific string.

# If True, remove LOCAL_PROJECT/outputs before running. Usually leave False when using SKIP_EXISTING.
CLEAR_LOCAL_OUTPUTS = False

print("Drive project:", DRIVE_PROJECT)
print("Local project:", LOCAL_PROJECT)
print("Drive output folder for this run:", DRIVE_RUN_OUTPUT_DIR)

Drive project: /content/drive/MyDrive/thesis_colab/model_experiments_colab
Local project: /content/model_experiments_colab
Drive output folder for this run: /content/drive/MyDrive/thesis_colab/outputs_all_baselines_colab_vscode/20260501_084747


In [2]:
# 2) Install/check dependencies and GPU status
import subprocess
import sys
import platform

INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    packages = [
        "transformers<5",
        "accelerate",
        "sentencepiece",
        "protobuf<6",
        "scipy",
        "scikit-learn",
        "peft",
        "bitsandbytes",
        "huggingface_hub",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu_name:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print("gpu_total_memory_gb:", round(props.total_memory / 1024**3, 2))
        print("bf16_supported:", torch.cuda.is_bf16_supported())
except Exception as exc:
    print("Could not inspect torch/GPU:", repr(exc))

# nvidia-smi is useful, but keep this Python-driven for VS Code/Colab compatibility.
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not available.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
torch: 2.10.0+cu128
cuda_available: True
gpu_name: Tesla T4
gpu_total_memory_gb: 14.56
bf16_supported: True


In [3]:
# 3) Mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Could not import/mount google.colab.drive. Are you connected to a Colab runtime?")
    raise

Mounted at /content/drive


In [4]:
# 4) Sync project from Google Drive to /content
# Rerun this cell whenever you update files in Drive and want /content to use the latest version.
import os
import shutil
from pathlib import Path

if not DRIVE_PROJECT.exists():
    raise FileNotFoundError(
        f"Drive project folder not found: {DRIVE_PROJECT}\n"
        "Create/upload your folder at this path, or edit DRIVE_PROJECT in cell 1."
    )

if LOCAL_PROJECT.exists():
    shutil.rmtree(LOCAL_PROJECT)

ignore = shutil.ignore_patterns(
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    ".pytest_cache",
    ".mypy_cache",
    ".ipynb_checkpoints",
    "wandb",
)

shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT, ignore=ignore)
os.chdir(LOCAL_PROJECT)

if CLEAR_LOCAL_OUTPUTS:
    outputs_dir = LOCAL_PROJECT / "outputs"
    if outputs_dir.exists():
        shutil.rmtree(outputs_dir)

print("Working directory:", Path.cwd())
print("Top-level files:")
for p in sorted(Path.cwd().iterdir()):
    print("-", p.name)

Working directory: /content/model_experiments_colab
Top-level files:
- data
- participant_kit
- run_experiment.py
- src


In [5]:
# 5) Sanity-check expected files before running
from pathlib import Path

required_paths = [
    "run_experiment.py",
    "src/data.py",
    "src/prompts.py",
    "src/models_flan.py",
    "src/models_deberta.py",
    "src/models_qwen.py",
    "src/models_gemma.py",
    "data/SHROOM_dev-v2/val.model-agnostic.json",
    "data/SHROOM_trial-v1.1/trial-v1.json",
    "participant_kit/check_output.py",
    "participant_kit/score.py",
]

missing = [p for p in required_paths if not Path(p).exists()]
if missing:
    print("Missing required files:")
    for p in missing:
        print("-", p)
    raise FileNotFoundError("Project structure check failed.")

print("Project structure looks good.")

Project structure looks good.


In [6]:
# 6) Model list: screenshot-only OOTB baselines
# These should match the completed baseline metadata files shown in your screenshot.
MODELS = [
    {'model_type': 'deberta', 'model_name': 'cross-encoder/nli-deberta-v3-base', 'run': True},
    {'model_type': 'deberta', 'model_name': 'cross-encoder/nli-deberta-v3-large', 'run': True},
    {'model_type': 'deberta', 'model_name': 'cross-encoder/nli-deberta-v3-small', 'run': True},
    {'model_type': 'deberta', 'model_name': 'cross-encoder/nli-deberta-v3-xsmall', 'run': True},

    {'model_type': 'flan', 'model_name': 'google/flan-t5-base', 'run': True},
    {'model_type': 'flan', 'model_name': 'google/flan-t5-large', 'run': True},
    {'model_type': 'flan', 'model_name': 'google/flan-t5-small', 'run': True},
    {'model_type': 'flan', 'model_name': 'google/flan-t5-xl', 'run': True},

    {'model_type': 'gemma', 'model_name': 'google/gemma-3-1b-it', 'run': True},
    {'model_type': 'gemma', 'model_name': 'google/gemma-3-4b-it', 'run': True},
    {'model_type': 'gemma', 'model_name': 'google/gemma-3-270m-it', 'run': True},

    {'model_type': 'deberta', 'model_name': 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli', 'run': True},

    {'model_type': 'qwen', 'model_name': 'Qwen/Qwen2.5-0.5B-Instruct', 'run': True},
    {'model_type': 'qwen', 'model_name': 'Qwen/Qwen2.5-1.5B-Instruct', 'run': True},
    {'model_type': 'qwen', 'model_name': 'Qwen/Qwen2.5-3B-Instruct', 'run': True},
    {'model_type': 'qwen', 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'run': True},

    {'model_type': 'deberta', 'model_name': 'sileod/deberta-v3-base-tasksource-nli', 'run': True},
]

print(f"Configured {len(MODELS)} models.")
for item in MODELS:
    print(f"- {item['model_type']}: {item['model_name']}")

Configured 17 models.
- deberta: cross-encoder/nli-deberta-v3-base
- deberta: cross-encoder/nli-deberta-v3-large
- deberta: cross-encoder/nli-deberta-v3-small
- deberta: cross-encoder/nli-deberta-v3-xsmall
- flan: google/flan-t5-base
- flan: google/flan-t5-large
- flan: google/flan-t5-small
- flan: google/flan-t5-xl
- gemma: google/gemma-3-1b-it
- gemma: google/gemma-3-4b-it
- gemma: google/gemma-3-270m-it
- deberta: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
- qwen: Qwen/Qwen2.5-0.5B-Instruct
- qwen: Qwen/Qwen2.5-1.5B-Instruct
- qwen: Qwen/Qwen2.5-3B-Instruct
- qwen: Qwen/Qwen2.5-7B-Instruct
- deberta: sileod/deberta-v3-base-tasksource-nli


In [7]:
# 7) Optional Hugging Face login
# Useful for gated models such as Gemma. In Colab, you can store a token in Secrets as HF_TOKEN.
LOGIN_TO_HF = False

if LOGIN_TO_HF:
    from huggingface_hub import login

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token is None:
        import getpass
        token = getpass.getpass("Paste Hugging Face token: ")

    login(token=token)
    print("Logged into Hugging Face.")
else:
    print("HF login skipped. Set LOGIN_TO_HF = True if a model requires access.")

HF login skipped. Set LOGIN_TO_HF = True if a model requires access.


In [8]:
# 8) Runner helpers: filtering, streamed subprocess logs, and Drive backup
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from datetime import datetime

def safe_filename(text: str) -> str:
    return (
        text.replace("/", "__")
        .replace("\\", "__")
        .replace(":", "_")
        .replace(" ", "_")
    )

def selected_models(models: list[dict]) -> list[dict]:
    only_types = {x.strip() for x in ONLY_TYPES if x.strip()} if ONLY_TYPES else set()
    only_contains = ONLY_CONTAINS.strip().lower()

    selected = []
    for item in models:
        if not item.get("run", True):
            continue
        if only_types and item["model_type"] not in only_types:
            continue
        if only_contains and only_contains not in item["model_name"].lower():
            continue
        selected.append(item)
    return selected

def backup_outputs_to_drive() -> None:
    DRIVE_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    src_outputs = LOCAL_PROJECT / "outputs"
    dst_outputs = DRIVE_RUN_OUTPUT_DIR / "outputs"

    if src_outputs.exists():
        if dst_outputs.exists():
            shutil.rmtree(dst_outputs)
        shutil.copytree(src_outputs, dst_outputs)
        print(f"Backed up outputs -> {dst_outputs}")
    else:
        print("No outputs directory yet; skipping outputs backup.")

def run_streamed(cmd: list[str], env: dict[str, str], log_path: Path) -> int:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Command:", " ".join(cmd))
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return process.wait()

def run_one_model(item: dict) -> dict:
    model_type = item["model_type"]
    model_name = item["model_name"]

    metadata_path = LOCAL_PROJECT / "outputs" / "metadata" / f"run__{safe_filename(model_name)}.json"
    if SKIP_EXISTING and metadata_path.exists():
        print(f"Skipping existing result: {model_name}")
        return {"model_name": model_name, "model_type": model_type, "status": "skipped_existing", "returncode": 0}

    notes = item.get("notes") or f"Colab VS Code OOTB baseline rerun; run_tag={RUN_TAG}"

    cmd = [
        sys.executable,
        "run_experiment.py",
        "--model-type", model_type,
        "--model-name", model_name,
        "--input-path", INPUT_PATH,
        "--seed", str(SEED),
        "--preview-n", str(PREVIEW_N),
        "--prompt-version", PROMPT_VERSION,
        "--notes", notes,
        "--warmup-path", WARMUP_PATH,
        "--warmup-n", str(WARMUP_N),
    ]

    env = os.environ.copy()
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    log_path = DRIVE_RUN_OUTPUT_DIR / "logs" / f"{safe_filename(model_name)}.log"

    print("\n" + "=" * 100)
    print(f"Running: {model_type} | {model_name}")
    print("=" * 100)

    if DRY_RUN:
        print("DRY_RUN=True, not executing.")
        return {"model_name": model_name, "model_type": model_type, "status": "dry_run", "returncode": 0}

    start = datetime.now().isoformat(timespec="seconds")
    returncode = run_streamed(cmd, env=env, log_path=log_path)
    end = datetime.now().isoformat(timespec="seconds")

    status = "success" if returncode == 0 else "failed"

    result = {
        "model_name": model_name,
        "model_type": model_type,
        "status": status,
        "returncode": returncode,
        "started_at": start,
        "ended_at": end,
        "log_path": str(log_path),
    }

    # Back up after every model so partial progress survives disconnects.
    backup_outputs_to_drive()

    # Clear CUDA cache between models.
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

    return result

In [9]:
# 9) Optional smoke/dry run
# Recommended first checks:
# - Set DRY_RUN=True to verify commands.
# - Or set ONLY_CONTAINS="flan-t5-small" and DRY_RUN=False for one real test.
print("DRY_RUN:", DRY_RUN)
print("ONLY_TYPES:", ONLY_TYPES)
print("ONLY_CONTAINS:", ONLY_CONTAINS)

selected = selected_models(MODELS)
print(f"Selected {len(selected)} model(s):")
for item in selected:
    print(f"- {item['model_type']}: {item['model_name']}")

DRY_RUN: False
ONLY_TYPES: ['qwen']
ONLY_CONTAINS: 7B
Selected 1 model(s):
- qwen: Qwen/Qwen2.5-7B-Instruct


In [10]:
# 10) Run selected models and back up after each one
DRIVE_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = DRIVE_RUN_OUTPUT_DIR / "run_manifest.json"
results = []

selected = selected_models(MODELS)
if not selected:
    raise ValueError("No models selected. Check ONLY_TYPES / ONLY_CONTAINS / run flags.")

for item in selected:
    result = run_one_model(item)
    results.append(result)

    with manifest_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "run_tag": RUN_TAG,
                "drive_project": str(DRIVE_PROJECT),
                "local_project": str(LOCAL_PROJECT),
                "drive_run_output_dir": str(DRIVE_RUN_OUTPUT_DIR),
                "input_path": INPUT_PATH,
                "warmup_path": WARMUP_PATH,
                "warmup_n": WARMUP_N,
                "seed": SEED,
                "results": results,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    if result["returncode"] != 0:
        print(f"FAILED: {item['model_name']} with return code {result['returncode']}")
        if not CONTINUE_ON_ERROR:
            break

print("\nBatch complete. Manifest:", manifest_path)
print(json.dumps(results, indent=2))


Running: qwen | Qwen/Qwen2.5-7B-Instruct
Command: /usr/bin/python3 run_experiment.py --model-type qwen --model-name Qwen/Qwen2.5-7B-Instruct --input-path data/SHROOM_dev-v2/val.model-agnostic.json --seed 42 --preview-n 2 --prompt-version support_prompt_v1 --notes Colab VS Code OOTB baseline rerun; run_tag=20260501_084747 --warmup-path data/SHROOM_trial-v1.1/trial-v1.json --warmup-n 10
`torch_dtype` is deprecated! Use `dtype` instead!
2026-05-01 08:49:50.680312: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading data from: data/SHROOM_dev-v2/val.model-agnostic.json
Loaded 499 examples.

--- Example 1 ---
id: None
task: DM
context: Resembling a weasel (in appearance).
hyp: Resembling or characteristic of a weasel.
label: Not Hallucination
p_h

In [11]:
# 11) Summarize metadata files found in outputs/metadata
import json
from pathlib import Path

metadata_dir = LOCAL_PROJECT / "outputs" / "metadata"
rows = []

if metadata_dir.exists():
    for path in sorted(metadata_dir.glob("run__*.json")):
        try:
            meta = json.loads(path.read_text(encoding="utf-8"))
        except Exception as exc:
            print("Could not read", path, exc)
            continue

        scores = meta.get("scores", {})
        comp = meta.get("computational_cost", {})
        rows.append({
            "model_type": meta.get("model_type"),
            "model_name": meta.get("model_name"),
            "parameter_count": comp.get("parameter_count"),
            "agnostic_acc": scores.get("agnostic_acc"),
            "agnostic_rho": scores.get("agnostic_rho"),
            "mean_latency_s": comp.get("mean_inference_latency_seconds_per_example"),
            "metadata_file": str(path),
        })

if not rows:
    print("No metadata rows found yet.")
else:
    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        display(df.sort_values(["model_type", "parameter_count"], na_position="last"))
    except Exception:
        for row in rows:
            print(row)

backup_outputs_to_drive()

,model_type,model_name,parameter_count,agnostic_acc,agnostic_rho,mean_latency_s,metadata_file
0,qwen,Qwen/Qwen2.5-7B-Instruct,7615616512,0.735471,0.624124,0.215969,/content/model_experiments_colab/outputs/metad...


Backed up outputs -> /content/drive/MyDrive/thesis_colab/outputs_all_baselines_colab_vscode/20260501_084747/outputs


In [31]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   83G   31G  74% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
tmpfs           6.4G  116K  6.4G   1% /var/colab
/dev/sda1       119G   86G   34G  72% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
drive            15G  7.2G  7.9G  48% /content/drive


In [32]:
!du -sh ~/.cache/huggingface/hub || true

39G	/root/.cache/huggingface/hub


In [33]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.2Gi        10Gi        12Mi       1.0Gi        11Gi
Swap:             0B          0B          0B


In [34]:
!nvidia-smi

Thu Apr 30 12:07:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             27W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [48]:
!rm -rf ~/.cache/huggingface/hub

In [ ]:
# 12) Optional: zip current /content outputs for manual retrieval
import shutil
from pathlib import Path

zip_base = Path("/content/shroom_all_baseline_outputs_colab_vscode")
zip_file = zip_base.with_suffix(".zip")

if zip_file.exists():
    zip_file.unlink()

outputs_dir = LOCAL_PROJECT / "outputs"
if outputs_dir.exists():
    shutil.make_archive(str(zip_base), "zip", root_dir=LOCAL_PROJECT, base_dir="outputs")
    print("Created:", zip_file)
else:
    print("No outputs directory to zip.")